In [ ]:
for id_block in range(num_blocks):

    position_start = len_prompt + id_block * block_length
    position_end = position_start + block_length
    block_mask_index = (x[:, position_start:position_end] == mask_id)  # (B, block_length)
    num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)  # (B, steps_per_block)

    ''' snapshot cache first round idx starts '''
    idx_refresh = get_empty_tensor2d_long(x)
    idx_denoising = torch.arange(position_start, position_end,dtype=torch.long, device=x.device)
    ''' snapshot cache first round idx ends '''

    idx_current = torch.cat([idx_refresh, idx_denoising])
    x_current = x[:, idx_current]

    output_snapshot = model(
        x_current,
        past_key_values=past_key_values,
        use_cache=helper_cache.use_cache,
        idx_current=idx_current,
        shape_target=(x.shape[0], position_end, -1)
    )

    logits_snapshot = output_snapshot.logits    # (B, L_current, H)
    logits_blk = logits_snapshot[:, idx_denoising] # (B, [blk]) TODO: check here

    # Mask and quota for this step (all tensor ops)
    mask_blk = (x[:, position_start:position_end] == mask_id)  # (B, block_length)
    blk_x = x[:, position_start:position_end]
    blk_y = y[:, position_start:position_end]
    blk_prob = probs_all[:, position_start:position_end]

    quota_0 = num_transfer_tokens[:, 0]  # (B,)
    blk_x0, blk_x0_p, transfer_idx_blk, idx_sorted_blk = get_transfer_index(
        logits_blk,
        temperature,
        remasking,
        mask_blk,
        blk_x,
        blk_y,
        quota_0
    )

    if is_eval:
        blk_x[transfer_idx_blk] = blk_y[transfer_idx_blk]
        blk_prob[transfer_idx_blk] = blk_x0_p[transfer_idx_blk]
    else:
        blk_x[transfer_idx_blk] = blk_x0[transfer_idx_blk]
    # end








    



    for step_in_block in range(steps_per_block):
        # Evaluate logits only for current block with cache
        if (x[:, position_start:position_end] == mask_id).sum() == 0:
            break
        # end

        # (B, [refresh|blk])
        past_key_values=output_current.past_key_values if helper_cache.use_cache else None # update past_key_values here



        ''' snapshot cache prepare for next round idx starts '''
        idx_refresh = get_denoise_idx_from_snapshot(transfer_idx_blk)
        idx_denoising = get_empty_tensor2d_long(x)
        ''' snapshot cache prepare for next round idx ends'''
    # end for step
# end for block
probs_all, y != mask_id